# **Q7: Intel Image Classification using Transfer Learning (ResNet)**
### **Program:** MSc/M.Tech in Artificial Intelligence
### **Course:** Computer Vision and Generative AI
### **Exam:** Final (27th June 2026)

This notebook implements an end-to-end image classification pipeline using **PyTorch** and **Transfer Learning (ResNet18)** to classify outdoor scene images into 6 categories: *buildings, forest, glacier, mountain, sea, and street*.

## **a. Data Preparation**
### **Step 1: Import Dependencies & Verify GPU**
We begin by importing required libraries for Deep Learning, data manipulation, and visualization. We also configure the execution to run on a **GPU** for accelerated execution.

In [ ]:
import os
import json
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
from PIL import Image

# Force GPU usage if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

### **Step 2: Define Data Augmentations and Create DataLoaders**
To prepare our dataset and prevent overfitting, we apply standard preprocessing:
- Resize to $224 \times 224$ pixels (required input dimension for ResNet)
- Add random transformations like `RandomHorizontalFlip`, `RandomRotation`, and `ColorJitter` to the training set.
- Normalize datasets using standard ImageNet mean and standard deviation values.

In [ ]:
# Define data transformations
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
}

# Update paths according to your environment layout
DATA_DIR = './'
TRAIN_DIR = os.path.join(DATA_DIR, 'seg_train/seg_train')
TEST_DIR = os.path.join(DATA_DIR, 'seg_test/seg_test')

try:
    # Load datasets using ImageFolder
    train_dataset = datasets.ImageFolder(root=TRAIN_DIR, transform=data_transforms['train'])
    test_dataset = datasets.ImageFolder(root=TEST_DIR, transform=data_transforms['val'])

    # Create DataLoaders
    BATCH_SIZE = 64
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    class_names = train_dataset.classes
    num_classes = len(class_names)

    print(f"Class Names: {class_names}")
    print(f"Number of training images: {len(train_dataset)}")
    print(f"Number of testing images: {len(test_dataset)}")
except Exception as e:
    print(f"Directory notice: Ensure dataset directories exist at {TRAIN_DIR} and {TEST_DIR}. Details: {e}")
    class_names = ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']
    num_classes = 6

## **b. Model Training using Transfer Learning**
### **Step 3: Setup Pretrained ResNet architecture**
We instantiate a standard pretrained **ResNet18** model, freeze its primary convolutional layers to lock feature representations, and re-create the final fully connected classifier layer to map features into our 6 unique categories.

In [ ]:
# Load pretrained ResNet18 backbone
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Freeze pretrained layers
for param in model.parameters():
    param.requires_grad = False

# Substitute final fully connected layer
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, num_classes)

# Send model to device
model = model.to(device)

# Specify Loss criteria and Optimizer targeting ONLY the classifier head
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

### **Step 4: Execute Model Training Loop**
We execute training across **10 epochs** to achieve robust metrics quickly within standard constraints.

In [ ]:
EPOCHS = 10
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

print("Starting Training...")
# Note: Execution requires local data structure populated. Here is an illustrative placeholder execution wrapper:
if 'train_loader' in globals():
    for epoch in range(EPOCHS):
        # --- Training Loop ---
        model.train()
        running_loss, running_corrects = 0.0, 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            _, preds = torch.max(outputs, 1)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)
        
        epoch_train_loss = running_loss / len(train_dataset)
        epoch_train_acc = (running_corrects.double() / len(train_dataset)).item()

        # --- Validation Loop ---
        model.eval()
        running_val_loss, running_val_corrects = 0.0, 0
        with torch.no_grad():
            for inputs, labels in test_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                _, preds = torch.max(outputs, 1)
                running_val_loss += loss.item() * inputs.size(0)
                running_val_corrects += torch.sum(preds == labels.data)
        
        epoch_val_loss = running_val_loss / len(test_dataset)
        epoch_val_acc = (running_val_corrects.double() / len(test_dataset)).item()
        
        history['train_loss'].append(epoch_train_loss)
        history['train_acc'].append(epoch_train_acc)
        history['val_loss'].append(epoch_val_loss)
        history['val_acc'].append(epoch_val_acc)
        
        print(f"Epoch {epoch+1}/{EPOCHS} -> Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.4f} || Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.4f}")
else:
    print("Skipping training loop execution block: setup dataset to evaluate locally.")

## **c. Model Evaluation and Inference**
### **Step 5: Visualize Performance Trends**

In [ ]:
if len(history['train_loss']) > 0:
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(history['train_loss'], label='Train Loss', color='crimson')
    plt.plot(history['val_loss'], label='Val Loss', color='navy')
    plt.title('Loss Curves')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history['train_acc'], label='Train Accuracy', color='crimson')
    plt.plot(history['val_acc'], label='Val Accuracy', color='navy')
    plt.title('Accuracy Curves')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No history metrics to display.")

### **How Overfitting is Detected From Training Curves**
Overfitting occurs when a model fits too closely to training subset particulars instead of generalized feature definitions. It is visually detectable via:
1. **Loss Divergence**: Training loss scales lower continuously, while validation loss ceases reducing and begins sloping upwards.
2. **Accuracy Gap Expansion**: Training accuracy climbs toward 100%, whereas evaluation accuracy stalls or scales negative, presenting a wide structural disparity.

### **Step 6: Evaluation Metrics (Confusion Matrix & Classification Report)**

In [ ]:
if 'test_loader' in globals():
    all_preds, all_labels = [], []
    model.eval()
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    print("\n=== Classification Report ===")
    print(classification_report(all_labels, all_preds, target_names=class_names))

    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.title('Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.show()
else:
    print("Dataloader dependencies unavailable for direct summary processing.")

### **Step 7: Utility Functions for Saving and Loading**

In [ ]:
def save_model(model, filepath="q7_intel_model.pth"):
    """Saves model state dictionary weights."""
    torch.save(model.state_dict(), filepath)
    print(f"Model saved successfully to {filepath}")

def load_model(filepath="q7_intel_model.pth", num_classes=6):
    """Loads saved model configuration onto backbone structure."""
    model_arch = models.resnet18()
    num_ftrs = model_arch.fc.in_features
    model_arch.fc = nn.Linear(num_ftrs, num_classes)
    model_arch.load_state_dict(torch.load(filepath, map_location=device))
    model_arch = model_arch.to(device)
    model_arch.eval()
    print(f"Model loaded successfully from {filepath}")
    return model_arch

# Test execution
save_model(model, "q7_intel_model.pth")

### **Step 8: Perform Target Image Inference**

In [ ]:
def predict_image(image_path, model_path="q7_intel_model.pth"):
    """Predicts the target class label for a single input image."""
    eval_model = load_model(model_path, num_classes=6)
    
    inference_transforms = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    
    try:
        img = Image.open(image_path).convert('RGB')
        input_tensor = inference_transforms(img).unsqueeze(0).to(device)
        
        with torch.no_grad():
            outputs = eval_model(input_tensor)
            probabilities = torch.nn.functional.softmax(outputs[0], dim=0)
            confidence, predicted_idx = torch.max(probabilities, 0)
            
        predicted_class = class_names[predicted_idx.item()]
        
        plt.imshow(img)
        plt.title(f"Prediction: {predicted_class} ({confidence.item()*100:.2f}%)")
        plt.axis('off')
        plt.show()
    except Exception as e:
        print(f"Inference error: Verify target file path. Details: {e}")

# Usage example:
# predict_image('seg_pred/101.jpg')